# Adım 5 — Mevzuat Asistanı (RAG)
4. adımda gördük: dil modelleri bilmedikleri yerde **akıcı biçimde uyduruyor**. Mevzuatta bu kabul edilemez.
**RAG (Retrieval-Augmented Generation)** ile model cevap vermeden önce ilgili maddeleri belgelerden **bulur**, sadece onlara dayanarak cevap verir ve **kaynağını gösterir**.

**Akış:**
1. Mevzuat belgelerini (PDF/Word) oku
2. **Madde madde** böl; her parça belge adı ve madde numarasını taşısın
3. İki arama yöntemi kur: **BM25** (kelime eşleşmesi) ve **BGE-M3** (anlam benzerliği); sonra ikisini birleştir
4. Aramayı test soruları ile **ölç** (doğru madde ilk 1 / ilk 5 sonuçta mı?)
5. **Kumru-2B** ile kaynaklı cevap üret; aynı soruyu RAG'siz sorup farkı gör

**Belgeleri hazırlama (bir kez):**
mevzuat.gov.tr otomatik indirmeye izin vermiyor (robots.txt), bu yüzden belgeleri elle indiriyoruz:
1. mevzuat.gov.tr'de belgeyi aç, **PDF** olarak indir.
2. Drive'da `yapay_zeka_projesi/mevzuat/` klasörüne koy.
3. Dosya adını anlaşılır yap; cevaplarda kaynak olarak bu ad görünecek. Önerilen başlangıç seti:
   - `Anayasa.pdf` (2709)
   - `1739_Milli_Egitim_Temel_Kanunu.pdf`
   - `657_Devlet_Memurlari_Kanunu.pdf`
   - İlgilendiğin MEB yönetmelikleri (ör. ortaöğretim kurumları, ek ders kararı)
   - İstersen büyük kanunlar da: Türk Medeni Kanunu (4721), CMK (5271), TCK (5237)… Binlerce madde sorun değil.
     Vektörler ilk seferde hesaplanıp Drive'a önbelleğe alınır; belgeler değişmedikçe tekrar hesaplanmaz.
     **Her yeni kanun için 5. bölümdeki test sorularına birkaç soru ekle**: kapsam genişledikçe karışma riski artar
     (ör. "izin" 657'de memur izni, Medeni Kanun'da veli izni).

**Uyarı:** Bu asistan hukuki danışmanlık yerine geçmez. Mevzuat değişir; cevaplar ancak klasördeki belgeler kadar günceldir.

**Çalışma zamanı:** `L4 GPU` önerilir (T4 de çalışır).

## 1. Kurulum

In [ ]:
%pip install -q "transformers>=4.51" accelerate sentence-transformers rank_bm25 pymupdf python-docx

In [ ]:
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # indirme çubuklarını gizle (defter temiz kalsın)

import re, gc, math, time, json, glob, unicodedata
from pathlib import Path
from collections import Counter
import numpy as np
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Cihaz:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0),
          f"| VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJE = "/content/drive/MyDrive/yapay_zeka_projesi"
except ImportError:
    PROJE = "./yapay_zeka_projesi"

MEVZUAT_KLASORU = f"{PROJE}/mevzuat"
os.makedirs(MEVZUAT_KLASORU, exist_ok=True)
dosyalar = sorted(glob.glob(f"{MEVZUAT_KLASORU}/*.pdf") + glob.glob(f"{MEVZUAT_KLASORU}/*.docx"))
print(f"\nMevzuat klasörü: {MEVZUAT_KLASORU}")
for d in dosyalar:
    print("  •", os.path.basename(d), f"({os.path.getsize(d) / 1e6:.1f} MB)")
if not dosyalar:
    print("  Klasör boş! Belgeleri Drive'da bu klasöre koyup hücreyi tekrar çalıştır.")

## 2. Belgeleri okuma ve temizleme
PDF'lerde her sayfada tekrar eden başlık/altbilgi, satır sonu bölünmeleri ve kelime ortasında tire ile bölünmeler olur. Bunları temizliyoruz.

Bazı PDF'lerde Türkçe harfler yanlış kodlamayla okunur: *"deðiþtirilmesi"* (ş → þ, ı → ý, ğ → ð, İ → Ý). Bu durumda arama, doğru yazılmış kelimeyi asla bulamaz. Defter bunu tespit edip düzeltir ve **⚠ uyarısı** verir. Uyarı görürsen belgenin daha temiz bir kopyasını (ör. mevzuat.gov.tr'den) bulmak en iyisidir.

**Taranmış PDF'ler** (fotokopi, arşiv taraması) metin değil görüntü içerir; bunlardan metin çıkmaz. Sayfaların çoğunda neredeyse hiç metin yoksa defter uyarı verir.

**Otomatik kontroller özeti:**
| Durum | Ne yapılıyor |
|---|---|
| Sayfa başlığı, altbilgi, sayfa numarası | Sık tekrar eden kısa satırlar silinir |
| Satır sonunda tireyle bölünmüş kelime | Birleştirilir |
| Bozuk Türkçe karakter (þ, ý, ð) | Düzeltilir, ⚠ uyarı |
| Taranmış (görüntü) PDF | ⚠ uyarı |
| Tek dosyada birden fazla belge | Madde numarası 1'e dönünce ayrılır, ⚠ uyarı (3. bölüm) |
| Roma rakamlı ve iki satırlı başlıklar | Doğru maddeye bağlanır (3. bölüm) |
| Atlanmış maddeler | Kontrol tablosunda gösterilir (3. bölüm) |

In [ ]:
import pymupdf
import docx

TARAMA_ESIGI = 50       # sayfa başına bundan az karakter: muhtemelen taranmış (görüntü) PDF. Taranmış sayfalardan genelde hiç metin çıkmaz; kısa ama gerçek sayfaları yanlışlıkla işaretlememek için düşük tutuldu.

def pdf_oku(yol):
    with pymupdf.open(yol) as belge:
        sayfalar = [sayfa.get_text() for sayfa in belge]
    bos = sum(len(s.strip()) < TARAMA_ESIGI for s in sayfalar)
    if sayfalar and bos / len(sayfalar) > 0.5:
        print(f"  ⚠ {Path(yol).name}: {len(sayfalar)} sayfanın {bos}'inde neredeyse hiç metin yok. "
              f"Taranmış (görüntü) bir PDF olabilir; OCR gerekir ya da metin içeren bir kopyasını bul.")
    return "\n".join(sayfalar)

def docx_oku(yol):
    return "\n".join(p.text for p in docx.Document(yol).paragraphs)

# Bazı PDF'lerde Türkçe harfler yanlış kod sayfasıyla okunur (mojibake): "deðiþtirilmesi" gibi.
# Türkçe mevzuat metinlerinde bu karakterler normalde geçmez; güvenle düzeltilebilir.
MOJIBAKE = str.maketrans({"þ": "ş", "Þ": "Ş", "ý": "ı", "Ý": "İ", "ð": "ğ", "Ð": "Ğ"})

def mojibake_onar(metin, ad=""):
    n = sum(metin.count(c) for c in "þÞýÝðÐ")
    if n:
        print(f"  ⚠ {ad}: {n:,} bozuk Türkçe karakter (þ/ý/ð…) bulundu ve düzeltildi.")
    return metin.translate(MOJIBAKE)

def temizle(metin, ad=""):
    metin = unicodedata.normalize("NFC", metin)
    metin = mojibake_onar(metin, ad)
    metin = metin.replace("\u00ad", "")                              # yumuşak tire
    metin = re.sub(r"(\w)-\n(\w)", r"\1\2", metin)                     # satır sonunda bölünmüş kelime
    satirlar = [s.strip() for s in metin.split("\n")]
    sayac = Counter(s for s in satirlar if s)
    # Sayfa başlık/altbilgileri: kısa ve çok sık tekrar eden satırlar
    tekrarlar = {s for s, n in sayac.items() if n >= 5 and len(s) < 80 and not re.match(r"(?i)^(madde|geçici madde|ek madde)\b", s)}
    satirlar = [s for s in satirlar if s not in tekrarlar and not re.fullmatch(r"\d{1,4}", s)]
    metin = "\n".join(satirlar)
    metin = re.sub(r"[ \t]+", " ", metin)
    metin = re.sub(r"\n{3,}", "\n\n", metin)
    return metin.strip()

def belge_adi(yol):
    return Path(yol).stem.replace("_", " ")

belgeler = {}
for d in dosyalar:
    ham = pdf_oku(d) if d.lower().endswith(".pdf") else docx_oku(d)
    belgeler[belge_adi(d)] = temizle(ham, belge_adi(d))
    print(f"{belge_adi(d):<45} {len(ham):>9,} → {len(belgeler[belge_adi(d)]):>9,} karakter")

## 3. Madde madde bölme
Türk mevzuatında maddeler `MADDE 1-`, `Madde 1 –`, `GEÇİCİ MADDE 2-`, `EK MADDE 3-` gibi başlar.
Her maddeyi ayrı bir parça yapıyoruz; çok uzun maddeleri (ör. tanımlar maddesi) örtüşen alt parçalara bölüyoruz.
**Bir maddenin başlığı genellikle bir önceki satırdadır** (ör. "Eğitim ve öğrenim hakkı ve ödevi"); onu da parçaya ekliyoruz, çünkü arama için çok değerli.

In [ ]:
MADDE_DESENI = re.compile(
    r"(?m)^[ \t]*((?:GEÇİCİ|Geçici|EK|Ek)?\s*(?:MADDE|Madde)\s+(\d+(?:/[A-Za-zÇĞİÖŞÜçğıöşü]+)?))\s*[-–—]"
)
AZAMI_PARCA = 1800      # karakter
ORTUSME = 200

def madde_etiketi(ham):
    # "GEÇİCİ MADDE 2" -> "Geçici Madde 2". Python'un .title() fonksiyonu Türkçe İ harfini bozduğu için elle yapıyoruz.
    numara = re.search(r"\d+(?:/[A-Za-zÇĞİÖŞÜçğıöşü]+)?", ham).group(0)
    if re.match(r"(?i)\s*(geçici|GEÇİCİ)", ham):
        return f"Geçici Madde {numara}"
    if re.match(r"(?i)\s*ek\s", ham):
        return f"Ek Madde {numara}"
    return f"Madde {numara}"

def maddelere_bol(ad, metin):
    eslesmeler = list(MADDE_DESENI.finditer(metin))
    parcalar = []
    if not eslesmeler:                       # madde yapısı yoksa düz parçalara böl
        for i in range(0, len(metin), AZAMI_PARCA - ORTUSME):
            parcalar.append({"belge": ad, "madde": f"bölüm {i // (AZAMI_PARCA - ORTUSME) + 1}",
                             "metin": metin[i:i + AZAMI_PARCA]})
        return parcalar
    # Her maddenin başlığını bul: maddeden hemen önceki kısa ve noktalama ile bitmeyen satır
    basliklar, baslik_baslangiclari = [], []
    for m in eslesmeler:
        onceki = metin[:m.start()].rstrip("\n ")
        satir_bas = onceki.rfind("\n") + 1
        satir = onceki[satir_bas:].strip()
        # 1739'daki "III – Yöneltme:" gibi Roma rakamlı kısa başlıklar iki noktayla bitebilir
        roma_basligi = re.match(r"^[IVXLC]+\s*[-–—]", satir) and len(satir) < 60
        if 0 < len(satir) < 120 and (not satir.endswith((".", ":", ";", ",")) or roma_basligi) and not MADDE_DESENI.match(satir):
            # İki satıra bölünmüş başlık: "…millî marşı" + "ve başkenti" (ikinci satır küçük harfle başlar)
            if satir[:1].islower() and satir_bas > 0:
                ust = onceki[:satir_bas - 1].rstrip("\n ")
                ust_bas = ust.rfind("\n") + 1
                ust_satir = ust[ust_bas:].strip()
                if 0 < len(ust_satir) < 120 and not ust_satir.endswith((".", ":", ";", ",")) and not MADDE_DESENI.match(ust_satir):
                    satir, satir_bas = ust_satir + " " + satir, ust_bas
            basliklar.append(satir); baslik_baslangiclari.append(satir_bas)
        else:
            basliklar.append(""); baslik_baslangiclari.append(m.start())

    # Numara sıfırlanma tespiti: "Madde 1" ikinci kez gelirse dosyada birden fazla belge vardır
    # (ör. Anayasa PDF'inin arkasına eklenmiş TBMM İçtüzüğü). İkinci kısım ayrı belge olarak etiketlenir.
    belge_no, son_numara, belge_adlari = 1, 0, []
    for m in eslesmeler:
        etiket = madde_etiketi(m.group(1))
        if etiket.startswith("Madde "):
            numara = int(re.match(r"Madde (\d+)", etiket).group(1))
            if numara == 1 and son_numara > 1:
                belge_no += 1
                print(f"  ⚠ {ad}: madde numaraları yeniden 1'den başladı → '{ad} (ek belge {belge_no})' olarak ayrıldı. "
                      f"Dosyada birden fazla belge olabilir; kontrol et.")
            son_numara = numara
        belge_adlari.append(ad if belge_no == 1 else f"{ad} (ek belge {belge_no})")
    for i, m in enumerate(eslesmeler):
        bas = m.start()
        # Madde, bir sonraki maddenin BAŞLIĞINDAN önce biter (başlık bir sonraki maddeye aittir)
        son = baslik_baslangiclari[i + 1] if i + 1 < len(eslesmeler) else len(metin)
        govde = metin[bas:son].strip()
        etiket = madde_etiketi(m.group(1))
        tam = (basliklar[i] + "\n" + govde) if basliklar[i] else govde
        if len(tam) <= AZAMI_PARCA:
            parcalar.append({"belge": belge_adlari[i], "madde": etiket, "metin": tam})
        else:
            for j, k in enumerate(range(0, len(tam), AZAMI_PARCA - ORTUSME)):
                parcalar.append({"belge": belge_adlari[i], "madde": f"{etiket} ({j + 1}. kısım)", "metin": tam[k:k + AZAMI_PARCA]})
    return parcalar

parcalar = []
for ad, metin in belgeler.items():
    p = maddelere_bol(ad, metin)
    parcalar.extend(p)
    print(f"{ad:<45} {len(p):>5} parça")
print(f"\nToplam: {len(parcalar):,} parça")

if parcalar:
    print("\n--- Örnek parça ---")
    ornek = parcalar[min(5, len(parcalar) - 1)]
    print(f"[{ornek['belge']} — {ornek['madde']}]\n{ornek['metin'][:600]}")

In [ ]:
# Kontrol: madde numaraları sırayla mı geliyor? Eksik/atlanmış madde, PDF okuma sorununa işaret eder.
for ad in sorted({p["belge"] for p in parcalar}):
    numaralar = set()
    for p in parcalar:
        m = re.match(r"Madde (\d+)(?: \(\d+\. kısım\))?$", p["madde"])     # kısımlara bölünen uzun maddeler de sayılır
        if p["belge"] == ad and m:
            numaralar.add(int(m.group(1)))
    if numaralar:
        eksik = sorted(set(range(1, max(numaralar) + 1)) - numaralar)
        print(f"{ad:<45} madde 1–{max(numaralar)} | eksik: {eksik[:15]}{' ...' if len(eksik) > 15 else ''}")

## 4. Arama yöntemleri
- **BM25:** Klasik kelime eşleşmesi. "657", "ek ders", "yıllık izin" gibi kesin terimlerde çok güçlü; eş anlamlıları bilmez.
- **BGE-M3:** Çok dilli bir **gömme (embedding)** modeli; metni anlamını temsil eden bir vektöre çevirir. "Öğretmen kaç gün izin kullanır?" sorusunu, kelimeler farklı olsa da "yıllık izin" maddesine bağlayabilir.
- **Hibrit (RRF):** İki sıralamayı birleştirir: her yöntemdeki sırasına göre puan verir (`1 / (60 + sıra)`), puanları toplar.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

import hashlib

def turkce_kucuk(metin):
    # Python'un .lower() fonksiyonu "I"yı "i"ye çevirir; Türkçede "ı" olmalı. "İ" de "i" olmalı.
    return metin.replace("I", "ı").replace("İ", "i").lower()

def kelimelere_ayir(metin):
    return re.findall(r"\w+", turkce_kucuk(metin))

bm25 = BM25Okapi([kelimelere_ayir(p["belge"] + " " + p["metin"]) for p in parcalar])

gomme_modeli = SentenceTransformer("BAAI/bge-m3", device=device)
gomme_metinleri = [f"{p['belge']} {p['madde']}\n{p['metin']}" for p in parcalar]

# Önbellek: parçalar değişmediyse vektörleri Drive'dan oku (büyük kanunlarda dakikalar kazandırır)
imza = hashlib.sha1("\x00".join(gomme_metinleri).encode("utf-8")).hexdigest()[:12]
ONBELLEK = f"{PROJE}/mevzuat_onbellek/vektorler_{imza}.npy"
os.makedirs(os.path.dirname(ONBELLEK), exist_ok=True)

t0 = time.time()
if os.path.exists(ONBELLEK):
    parca_vektorleri = np.load(ONBELLEK)
    print(f"Vektörler önbellekten okundu ({os.path.basename(ONBELLEK)})")
else:
    parca_vektorleri = gomme_modeli.encode(gomme_metinleri, batch_size=16, normalize_embeddings=True,
                                           show_progress_bar=True)
    np.save(ONBELLEK, parca_vektorleri)
    print(f"Vektörler hesaplandı ve önbelleğe kaydedildi ({os.path.basename(ONBELLEK)})")
print(f"{len(parcalar):,} parça | vektör boyutu: {parca_vektorleri.shape[1]} | {time.time() - t0:.0f} sn")

def ara_bm25(soru, k=5):
    puanlar = bm25.get_scores(kelimelere_ayir(soru))
    return list(np.argsort(-puanlar)[:k])

def ara_vektor(soru, k=5):
    v = gomme_modeli.encode([soru], normalize_embeddings=True)[0]
    return list(np.argsort(-(parca_vektorleri @ v))[:k])

def ara_hibrit(soru, k=5, aday=30, sabit=60):
    puan = Counter()
    for sira, i in enumerate(ara_bm25(soru, aday)):
        puan[i] += 1 / (sabit + sira)
    for sira, i in enumerate(ara_vektor(soru, aday)):
        puan[i] += 1 / (sabit + sira)
    return [i for i, _ in puan.most_common(k)]

def goster(soru, yontem=ara_hibrit, k=3):
    print(f"Soru: {soru}")
    for sira, i in enumerate(yontem(soru, k), 1):
        p = parcalar[i]
        print(f"  {sira}. [{p['belge']} — {p['madde']}] {p['metin'][:150].replace(chr(10), ' ')}...")

goster("Öğretmenlerin yıllık izin süresi ne kadardır?")

## 5. Aramayı ölçme
Önceki adımlardaki gibi bir **sayısal ölçü** kuruyoruz: her test sorusu için doğru maddenin hangi belgede ve hangi numarada olduğunu biliyoruz.
- **İsabet@1:** Doğru madde **ilk** sonuçta mı?
- **İsabet@5:** Doğru madde **ilk 5** sonuçta mı? (Kumru'ya 5 madde vereceğimiz için asıl önemli olan bu.)

Aşağıdaki sorular önerilen başlangıç setine (Anayasa, 1739, 657) göre hazırlandı. **Kendi yönetmeliklerin için de soru ekle**; ölçüm ancak test soruları kadar güvenilir. `belge` alanı dosya adında geçen bir ifadedir (büyük/küçük harf fark etmez).

In [ ]:
TEST_SORULARI = [
    {"soru": "Türkiye Cumhuriyeti'nin başkenti neresidir?",                     "belge": "anayasa", "madde": "3"},
    {"soru": "Milletvekili seçilebilmek için kaç yaşını doldurmak gerekir?",    "belge": "anayasa", "madde": "76"},
    {"soru": "Kimse eğitim ve öğrenim hakkından yoksun bırakılabilir mi?",       "belge": "anayasa", "madde": "42"},
    {"soru": "Türk milli eğitiminin genel amaçları nelerdir?",                   "belge": "1739",    "madde": "2"},
    {"soru": "Eğitimde fırsat ve imkân eşitliği nasıl sağlanır?",                "belge": "1739",    "madde": "8"},
    {"soru": "Eğitimde laiklik ilkesi neyi ifade eder?",                         "belge": "1739",    "madde": "12"},
    {"soru": "Devlet memurlarının haftalık çalışma süresi kaç saattir?",         "belge": "657",     "madde": "99"},
    {"soru": "Memurların yıllık izin süresi ne kadardır?",                       "belge": "657",     "madde": "102"},
]

def dogru_mu(parca, beklenen):
    # "(ek belge 2)" gibi ayrılmış kısımlar asıl belge sayılmaz (ör. Anayasa PDF'indeki İçtüzük)
    if beklenen["belge"].lower() not in parca["belge"].lower() or "(ek belge" in parca["belge"]:
        return False
    m = re.match(r"(?:Geçici |Ek )?Madde (\d+)", parca["madde"])
    return bool(m) and m.group(1) == beklenen["madde"] and not parca["madde"].startswith(("Geçici", "Ek"))

def olc(yontem, sorular, k=5):
    s1 = s5 = gecerli = 0
    for t in sorular:
        if not any(t["belge"].lower() in ad.lower() for ad in belgeler):
            continue                                   # o belge klasörde yoksa atla
        gecerli += 1
        sonuc = [parcalar[i] for i in yontem(t["soru"], k)]
        s1 += dogru_mu(sonuc[0], t)
        s5 += any(dogru_mu(p, t) for p in sonuc)
    return gecerli, s1, s5

print(f"{'Yöntem':<10} {'Soru':>5} {'İsabet@1':>10} {'İsabet@5':>10}")
for ad, yontem in [("BM25", ara_bm25), ("Vektör", ara_vektor), ("Hibrit", ara_hibrit)]:
    n, s1, s5 = olc(yontem, TEST_SORULARI)
    if n:
        print(f"{ad:<10} {n:>5} {s1 / n:>9.0%} {s5 / n:>10.0%}")
    else:
        print("Test sorularının belgeleri klasörde yok; TEST_SORULARI'nı kendi belgelerine göre düzenle.")
        break

In [ ]:
# Hangi sorular kaçırıldı? (Hibrit, ilk 5)
for t in TEST_SORULARI:
    if not any(t["belge"].lower() in ad.lower() for ad in belgeler):
        continue
    sonuc = [parcalar[i] for i in ara_hibrit(t["soru"], 5)]
    if not any(dogru_mu(p, t) for p in sonuc):
        print(f"✗ {t['soru']}  (beklenen: {t['belge']} m.{t['madde']})")
        for p in sonuc[:3]:
            print(f"     bulunan: [{p['belge']} — {p['madde']}]")
print("Kontrol bitti.")

## 6. Cevap üretme: Kumru-2B (talimatlı)
4. adımda **Base** modeli kullandık; o sadece metni devam ettirir. Soru cevaplamak için **talimat eğitimi almış** `vngrs-ai/Kumru-2B` kullanıyoruz.
Modele bulunan maddeleri veriyor ve kurallar koyuyoruz: **sadece bu maddelere dayan, cevap yoksa söyle.**

**Denemelerden çıkan üç ders (küçük modellerle çalışırken):**
1. **Model kaynak göstermeyi talimatla güvenilir şekilde yapmıyor.** Kurallarda "madde numarasını belirt" dememize rağmen Kumru cevabı doğru verip kaynağı atladı.
2. **Örnekli talimat tehlikeli.** Biçimi göstermek için istem içine bir örnek cevap ("Türkiye Devleti'nin başkenti Ankara'dır") koyduğumuzda model, memur çalışma süresi sorusuna bile **bu örneği kopyaladı.**
3. **Çözüm: işbölümü.** Model yalnızca cevabı yazar; cevaba biz "Cevap:" diye başlarız (**önden doldurma**) ki doğrudan konuya girsin. **Kaynağı kod bulur:** cevap vektöre çevrilir ve aranan maddelerden hangisine en çok benzediğine bakılır. Model yanlış bir madde numarası uyduramaz.

**Eşleşme puanı** cevabın doğruluğunu değil, maddeye ne kadar yakın ifade edildiğini ölçer. Kısa ama doğru bir cevabın puanı düşük olabilir; düşük puan "dikkatle bak" demektir.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

CEVAP_MODELI = "vngrs-ai/Kumru-2B"
c_tok = AutoTokenizer.from_pretrained(CEVAP_MODELI)
c_model = AutoModelForCausalLM.from_pretrained(CEVAP_MODELI, torch_dtype=torch.float32,
                                               low_cpu_mem_usage=True, device_map=device).eval()
print("Cevap modeli yüklendi:", CEVAP_MODELI)

KURALLAR = (
    "Sen Türk mevzuatı konusunda yardımcı bir asistansın. "
    "Soruyu YALNIZCA aşağıda verilen mevzuat maddelerine dayanarak, kısa ve açık cevapla. "
    "Verilen maddelerde cevap yoksa 'Verilen mevzuatta bu sorunun cevabını bulamadım.' de."
)   # Bilerek örnek cevap YOK: küçük model örneği kopyalıyor.

def mesaj_hazirla(soru, bulunan):
    kaynaklar = "\n\n".join(f"[{p['belge']} — {p['madde']}]\n{p['metin']}" for p in bulunan)
    return f"{KURALLAR}\n\n### Mevzuat maddeleri\n{kaynaklar}\n\n### Soru\n{soru}"

@torch.no_grad()
def uret(icerik, azami=400, on_ek="Cevap:"):
    # Önden doldurma: cevaba biz başlıyoruz, model doğrudan cevaba giriyor
    istem = c_tok.apply_chat_template([{"role": "user", "content": icerik}],
                                      add_generation_prompt=True, tokenize=False) + on_ek
    girdi = c_tok(istem, return_tensors="pt", add_special_tokens=False).to(device)
    cikti = c_model.generate(**girdi, max_new_tokens=azami, do_sample=False, repetition_penalty=1.1,
                             pad_token_id=c_tok.eos_token_id)
    return c_tok.decode(cikti[0, girdi["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def kaynak_bul(cevap, bulunan):
    # Kaynağı model değil kod belirler: cevaba anlamca en yakın madde
    v = gomme_modeli.encode([cevap], normalize_embeddings=True)[0]
    benzerlik = gomme_modeli.encode([p["metin"] for p in bulunan], normalize_embeddings=True) @ v
    en_iyi = int(np.argmax(benzerlik))
    return bulunan[en_iyi], float(benzerlik[en_iyi])

def soru_ilgisi(soru):
    # Soru, belgelerdeki herhangi bir maddeye ne kadar benziyor? (en yüksek benzerlik)
    v = gomme_modeli.encode([soru], normalize_embeddings=True)[0]
    return float((parca_vektorleri @ v).max())

# Eşikler 8. bölümdeki ölçüme göre seçildi (ilgili sorular 0,65–0,76; ilgisizler 0,44–0,50).
# Test seti büyüdükçe yeniden ayarlanmalı.
ILGI_ESIGI = 0.58       # bunun altındaki sorular modele hiç gönderilmez
KAYNAK_ESIGI = 0.60     # cevap-kaynak eşleşmesi bunun altındaysa uyarı verilir
KAPSAM = ", ".join(sorted({p["belge"] for p in parcalar}))

def sor(soru, k=3, aranan_kaynaklari_goster=False):
    """Üç katmanlı güvenlik:
    1) Soru belgelerle ilgisizse model çalıştırılmaz.
    2) Model 'bulamadım' derse kaynak gösterilmez.
    3) Cevap kaynağa yeterince benzemiyorsa uyarı verilir.
    Son katman şeffaflık: dayanılan maddenin metni gösterilir, kullanıcı kendisi doğrular."""
    print(f"SORU: {soru}\n")
    ilgi = soru_ilgisi(soru)
    if ilgi < ILGI_ESIGI:
        print(f"CEVAP: Belgelerde bu konuyla ilgili bir madde bulunamadı. (soru ilgisi: {ilgi:.2f})")
        print(f"Kapsam: {KAPSAM}\n" + "-" * 80)
        return None
    bulunan = [parcalar[i] for i in ara_hibrit(soru, k)]
    cevap = uret(mesaj_hazirla(soru, bulunan))
    print(f"CEVAP: {cevap}")
    if "bulamadım" in cevap.lower():
        # "Bulamadım" cevabına kaynak eklemek, cevabı bir maddeye dayanıyormuş gibi gösterir: yanıltıcı
        print("KAYNAK: — (cevap mevzuatta bulunamadı)")
    else:
        kaynak, benzerlik = kaynak_bul(cevap, bulunan)
        uyari = "" if benzerlik >= KAYNAK_ESIGI else "   ⚠ Kaynak doğrulanamadı, dikkatle değerlendirin."
        print(f"KAYNAK: {kaynak['belge']}, {kaynak['madde']}  (eşleşme: {benzerlik:.2f}){uyari}")
        print(f"  » {kaynak['metin'][:300].replace(chr(10), ' ')}…")       # kullanıcı kendisi doğrulayabilsin
    if aranan_kaynaklari_goster:
        print("Aranan kaynaklar:", "; ".join(f"{p['belge']} — {p['madde']}" for p in bulunan))
    print(f"Kapsam: {KAPSAM}\n" + "-" * 80)
    return cevap

_ = sor("Devlet memurlarının haftalık çalışma süresi kaç saattir?")
_ = sor("Memurların yıllık izin süresi ne kadardır?")
_ = sor("Öğretmenlerin yaz tatilinde izin hakkı var mıdır?")

## 7. RAG'li ve RAG'siz cevap karşılaştırması
Aynı soruyu kaynak vermeden de soruyoruz. Model kendi "hafızasından" cevap vermek zorunda kalınca neler uydurduğuna bak:
yanlış madde numaraları, var olmayan hükümler, yürürlükten kalkmış bilgiler.

In [ ]:
KARSILASTIRMA = [
    "Memurların yıllık izin süresi ne kadardır?",
    "Milletvekili seçilebilmek için kaç yaşını doldurmak gerekir?",
    "Eğitimde laiklik ilkesi neyi ifade eder?",
]
for soru in KARSILASTIRMA:
    print(f"SORU: {soru}")
    print("\n— RAG'SİZ (model kendi hafızasından) —")
    print(uret(f"{soru}\nCevabında dayandığın kanun ve madde numarasını belirt.", on_ek=""))
    bulunan = [parcalar[i] for i in ara_hibrit(soru, 3)]
    cevap = uret(mesaj_hazirla(soru, bulunan))
    print("\n— RAG'Lİ —")
    print(cevap)   # Not: ilk çalıştırmada laiklik sorusunda RAG'li cevap da yanlıştı (bkz. aşağıdaki not)
    if "bulamadım" not in cevap.lower():
        kaynak, benzerlik = kaynak_bul(cevap, bulunan)
        print(f"KAYNAK: {kaynak['belge']}, {kaynak['madde']}  (eşleşme: {benzerlik:.2f})")
    print("=" * 80)

**İlk çalıştırmanın sonuçları:**
- **Yıllık izin:** RAG'siz cevap **doğru maddeyi** gösterdi (657, Madde 102) ama içeriği uydurdu ("en fazla 3 yıl / 4 yıl"). En tehlikeli halüsinasyon türü: doğru kaynak numarası güven veriyor, içerik yanlış. RAG'li cevap doğruydu (20 / 30 gün).
- **Milletvekili yaşı:** İki cevap da doğruydu; çok bilinen bir bilgi.
- **Laiklik:** **RAG'li cevap da yanlıştı.** Model laikliği "ayrım gözetmeksizin herkese açık olma, fırsat eşitliği" diye tanımladı; bu ifadeler 1739'un başka maddelerinden geliyor. Doğru madde (12) ilk 5 sonuçtaydı: **arama başarılı oldu, cevap üretimi başarısız oldu.** Küçük model, birbirine yakın konudaki beş maddeyi birlikte okurken karıştırdı. Doğru belgeyi bulmak, doğru cevabı garanti etmez.

### Kaç kaynak vermeli? (k deneyi)
Laiklik sorusunda arama doğru maddeyi (1739, Madde 12) **birinci sıraya** koymuştu; yanlış cevap aramadan değil, üretimden geliyordu. Aynı soruyu 5, 3 ve 1 maddeyle soruyoruz ve `kaynak_bul` ile modelin hangi maddeye dayandığına bakıyoruz.

**İlk çalıştırmanın sonucu:**
| k | Modelin dayandığı madde | Cevap |
|---|---|---|
| 5 | 1739, Madde 4 (genellik ve eşitlik) | Yanlış |
| 3 | 1739, Madde 12 (laiklik) | Doğru maddeye dayanıyor |
| 1 | 1739, Madde 12 | Doğru maddeye dayanıyor |

Küçük model fazla bağlamda dikkatini dağıtıyor: birinci sıradaki doğru maddeyi bırakıp ikinciye yöneldi. **Varsayılan k=3 seçildi**: isabeti koruyor, dikkati dağıtmıyor. k=1 ise ilk sonuç yanıldığında cevabı tamamen kaçırır.

In [ ]:
soru = "Eğitimde laiklik ilkesi neyi ifade eder?"
bulunan = [parcalar[i] for i in ara_hibrit(soru, 5)]
print("Aranan:", [f"{p['belge'][:4]} {p['madde']}" for p in bulunan])
for k in (5, 3, 1):
    cevap = uret(mesaj_hazirla(soru, bulunan[:k]))
    kaynak, benzerlik = kaynak_bul(cevap, bulunan[:k])
    print(f"\nk={k}: {cevap[:200]}\n  → {kaynak['belge'][:4]} {kaynak['madde']} ({benzerlik:.2f})")

## 8. Kapsam dışı sorular ve güvenlik katmanları
İyi bir mevzuat asistanı, belgelerde olmayan bir soruya **uydurmak yerine** bilmediğini söylemeli.

İlk denemede Kumru futbol kontenjanı sorusuna "bulamadım" demedi; genel bir cevap verdi. Kod da bu cevaba **657, Madde 77**'yi kaynak olarak ekledi. Güvenlik kararını modelin ifadesine bırakmamak gerekiyor.

**1. katman — soru ilgisi:** Soru, belgelerdeki herhangi bir maddeye yeterince benzemiyorsa modeli hiç çalıştırmıyoruz. Eşiği tahmin etmek yerine ölçüyoruz:

In [ ]:
ilgili = ["Devlet memurlarının haftalık çalışma süresi kaç saattir?",
          "Memurların yıllık izin süresi ne kadardır?",
          "Öğretmenlerin yaz tatilinde izin hakkı var mıdır?",
          "Eğitimde laiklik ilkesi neyi ifade eder?",
          "Milletvekili seçilebilmek için kaç yaşını doldurmak gerekir?",
          "Okul öncesi eğitim zorunlu mudur?"]
ilgisiz = ["Türkiye'de futbol kulüplerinin yabancı oyuncu kontenjanı kaçtır?",
           "Kredi kartı asgari ödeme oranı nedir?",
           "Ankara'da yarın hava nasıl olacak?",
           "Trafik cezası itiraz süresi kaç gündür?"]
print("İLGİLİ SORULAR")
for s in ilgili:
    print(f"  {soru_ilgisi(s):.3f}  {s}")
print("İLGİSİZ SORULAR")
for s in ilgisiz:
    print(f"  {soru_ilgisi(s):.3f}  {s}")
print(f"\nKullanılan eşik: {ILGI_ESIGI}")

**Ölçüm sonucu (ilk çalıştırma):** İlgili sorular 0,654–0,756; futbol, kredi kartı ve hava durumu 0,438–0,498. Aradaki boşluğa **0,58** eşiği konuldu.
**Ama:** "Trafik cezası itiraz süresi" (hukuki ama kapsam dışı) **0,666** aldı; ilgili laiklik sorusundan (0,654) bile yüksek. Hukuki dille sorulan kapsam dışı soruları tek bir eşikle ayırmak mümkün değil.

**Bu yüzden üç katman + şeffaflık:**
1. Soru ilgisi < 0,58 → model çalıştırılmaz.
2. Model "bulamadım" derse → kaynak gösterilmez.
3. Cevap–kaynak eşleşmesi < 0,60 → "⚠ Kaynak doğrulanamadı" uyarısı.
4. **Son katman, şeffaflık:** Dayanılan maddenin metni ve kapsamdaki belgeler her cevapta gösterilir.

İlk denemede trafik cezası sorusu 1. katmanı geçti; model 657'nin disiplin zamanaşımı maddesini (Madde 135) gerçek bir ifadeyle aktardı, eşleşme yüksek çıktı (0,65) ve uyarı devreye girmedi. **Cevap uydurma değildi, başka bir soruya aitti.** Hiçbir otomatik kontrol bunu yakalayamadı; ama altında gösterilen madde metni ve kapsam satırı, kullanıcının bunu saniyeler içinde fark etmesini sağlar.

In [ ]:
for s in ["Türkiye'de futbol kulüplerinin yabancı oyuncu kontenjanı kaçtır?",
          "Trafik cezası itiraz süresi kaç gündür?",
          "Memurların yıllık izin süresi ne kadardır?"]:
    _ = sor(s)

## 9. Kendi sorularını sor
`sor("...")` ile istediğin soruyu sorabilirsin. Cevabın altındaki **Aranan kaynaklar** listesine bakarak doğru maddelerin bulunup bulunmadığını kontrol et.

In [ ]:
_ = sor("Okul öncesi eğitim zorunlu mudur?")

## 10. Alıştırmalar
1. **Kendi yönetmeliklerin:** Klasöre 2–3 MEB yönetmeliği ekle ve `TEST_SORULARI`'na her biri için 3–5 soru yaz. Hangi arama yöntemi kazanıyor?
2. **k değeri:** Varsayılan k=3 (bkz. 7. bölüm). `sor(..., k=1)` ve `k=5` ile farklı sorularda dene; laiklikteki karışma başka sorularda da oluyor mu?
3. **Kurallar:** `KURALLAR` metnini değiştir (ör. "maddeyi aynen alıntıla"). Cevapların biçimi nasıl değişiyor?
4. **Zor sorular:** İki maddeyi birleştirmeyi gerektiren sorular sor (ör. "Memur olan bir öğretmenin hem çalışma süresi hem izin hakkı nedir?").

## Sonuç tablosu
| Yöntem | İsabet@1 | İsabet@5 |
|---|---|---|
| BM25 |  |  |
| Vektör (BGE-M3) |  |  |
| Hibrit (RRF) |  |  |